# BanglishRev — Content-Based Recommender Notebook

**Pipeline:**
1. Load JSON with `review_text`
2. Deduplication
3. Convergent k-core filtering
4. 3-way temporal split
5. EDA (analyses 1, 3, 5, 6 + duplicates + tier breakdown)
6. Sparse matrix + index maps
7. Item profiles (TF-IDF, char n-gram, 3-tier fallback)
8. User profiles (mean TF-IDF)
9. ContentBasedRecommender class
10. Fit
11. Evaluate (full test set, K = 5, 10, 15, 30, 50)
12. Segmented evaluation by user script
13. Sample recommendation demonstration
14. Save results

In [ ]:
# ============================================================
# CELL 1 | Install dependencies
# ============================================================
!pip install scikit-learn scipy pandas numpy tabulate tqdm --quiet
print('✅ Dependencies ready.')

In [ ]:
# ============================================================
# CELL 2 | Imports
# ============================================================
import gc
import gzip
import json
import math
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from tabulate import tabulate
from tqdm import tqdm

warnings.filterwarnings('ignore')
print('✅ Imports done.')

In [ ]:
# ============================================================
# CELL 3 | Load JSON with review_text
# ============================================================

import orjson

DATA_PATH = 'BanglishRev.json.gz'   # ← update file path

print(f'⏳ Loading {DATA_PATH} …')
t0 = time.time()

open_fn = gzip.open if DATA_PATH.endswith('.gz') else open
mode    = 'rb'

with open_fn(DATA_PATH, mode) as f:
    raw_data = orjson.loads(f.read())

print(f'   Loaded {len(raw_data):,} products in {time.time()-t0:.1f}s')

# ── Flatten ───────────────────────────────────────────────
all_rows = []
n_skipped = 0

for product in tqdm(raw_data, desc='Flattening'):
    pid        = str(product.get('Product ID', ''))
    category   = product.get('Category',        'Unknown')
    parent_cat = product.get('Parent Category', 'Unknown')
    root_cat   = product.get('Root Category',   'Unknown')

    if not pid:
        n_skipped += 1
        continue

    for review in product.get('Reviews') or []:
        buyer_id = review.get('Buyer ID')
        rating   = review.get('Current Rating')

        if buyer_id is None or rating is None:
            n_skipped += 1
            continue

        try:
            rating = float(rating)
        except (ValueError, TypeError):
            n_skipped += 1
            continue

        all_rows.append({
            'user_id':     str(buyer_id),
            'product_id':  pid,
            'rating':      rating,
            'category':    str(category)   if category   else 'Unknown',
            'parent_cat':  str(parent_cat) if parent_cat else 'Unknown',
            'root_cat':    str(root_cat)   if root_cat   else 'Unknown',
            'timestamp':   review.get('Review Date'),
            # ── New vs v2 ─────────────────────────────────
            'review_text': review.get('Review Content') or '',
        })

del raw_data
gc.collect()

df_raw = pd.DataFrame(all_rows)
del all_rows
gc.collect()

# ── Basic dtype cleanup ───────────────────────────────────
df_raw['rating']      = pd.to_numeric(df_raw['rating'], errors='coerce')
df_raw['timestamp']   = pd.to_datetime(df_raw['timestamp'], errors='coerce')
df_raw['review_text'] = df_raw['review_text'].fillna('').astype(str)
df_raw = df_raw.dropna(subset=['user_id', 'product_id', 'rating', 'timestamp'])
df_raw['rating'] = df_raw['rating'].clip(1, 5).astype('float32')

for col in ['category', 'parent_cat', 'root_cat']:
    df_raw[col] = df_raw[col].astype('category')

print(f'\n✅ Loaded in {time.time()-t0:.1f}s')
print(f'   Rows            : {len(df_raw):,}')
print(f'   Skipped         : {n_skipped:,}')
print(f'   Unique users    : {df_raw["user_id"].nunique():,}')
print(f'   Unique items    : {df_raw["product_id"].nunique():,}')
print(f'   Rating range    : [{df_raw["rating"].min()}, {df_raw["rating"].max()}]')
print(f'   Reviews w/ text : {(df_raw["review_text"] != "").sum():,} '
      f'({(df_raw["review_text"] != "").mean()*100:.1f}%)')
df_raw.head(3)

In [ ]:
# ============================================================
# CELL 4 | Deduplication
# ============================================================

n_before = len(df_raw)

# Step 1 — exact duplicates
df_raw = df_raw.drop_duplicates()
print(f'After exact dedup     : {len(df_raw):,}  (removed {n_before - len(df_raw):,})')

# Step 2 — keep highest-rated interaction per (user, item) pair
df_raw = (
    df_raw
    .sort_values('rating', ascending=False)
    .drop_duplicates(subset=['user_id', 'product_id'], keep='first')
    .reset_index(drop=True)
)
print(f'After user-item dedup : {len(df_raw):,}  (removed {n_before - len(df_raw):,} total)')
print(f'Unique users          : {df_raw["user_id"].nunique():,}')
print(f'Unique items          : {df_raw["product_id"].nunique():,}')

In [ ]:
# ============================================================
# CELL 5 | Convergent iterative k-core filtering
# ============================================================

MIN_USER_INTERACTIONS = 10
MIN_ITEM_INTERACTIONS = 3

def iterative_filter(df, min_user=10, min_item=3):
    iteration    = 0
    previous_size = -1

    while previous_size != len(df):
        previous_size = len(df)
        iteration += 1

        user_counts = df['user_id'].value_counts()
        df = df[df['user_id'].isin(user_counts[user_counts >= min_user].index)]

        item_counts = df['product_id'].value_counts()
        df = df[df['product_id'].isin(item_counts[item_counts >= min_item].index)]

        print(f'  Pass {iteration}: {len(df):,} rows | '
              f'{df["user_id"].nunique():,} users | '
              f'{df["product_id"].nunique():,} items')

    return df.reset_index(drop=True)


print(f'Before filtering: {len(df_raw):,} rows')
df_filtered = iterative_filter(
    df_raw,
    min_user=MIN_USER_INTERACTIONS,
    min_item=MIN_ITEM_INTERACTIONS
)
print(f'\n✅ Final: {len(df_filtered):,} rows '
      f'({len(df_filtered)/len(df_raw)*100:.1f}% retained)')

del df_raw
gc.collect()

In [ ]:
# ============================================================
# CELL 6 | 3-way temporal split
# ============================================================

def temporal_split(df):
    df = df.sort_values(['user_id', 'timestamp'], na_position='last')
    train_list, val_list, test_list = [], [], []

    for user_id, group in tqdm(df.groupby('user_id', sort=False),
                               desc='Splitting', leave=False):
        if len(group) < 3:
            continue
        train_list.append(group.iloc[:-2])
        val_list.append(group.iloc[[-2]])
        test_list.append(group.iloc[[-1]])

    return (
        pd.concat(train_list, ignore_index=True),
        pd.concat(val_list,   ignore_index=True),
        pd.concat(test_list,  ignore_index=True),
    )


train_df, val_df, test_df = temporal_split(df_filtered)

# ── Leakage assertion ─────────────────────────────────────
train_pairs = set(zip(train_df['user_id'], train_df['product_id']))
val_pairs   = set(zip(val_df['user_id'],   val_df['product_id']))
test_pairs  = set(zip(test_df['user_id'],  test_df['product_id']))

assert len(val_pairs  & test_pairs)  == 0, 'Val/test leakage detected'
assert len(train_pairs & test_pairs) == 0, 'Train/test leakage detected'

print(f'✅ Split complete — no leakage')
print(f'   Train : {len(train_df):,} rows  |  {train_df["user_id"].nunique():,} users')
print(f'   Val   : {len(val_df):,}  rows  |  {val_df["user_id"].nunique():,} users')
print(f'   Test  : {len(test_df):,}  rows  |  {test_df["user_id"].nunique():,} users')

In [ ]:
# ============================================================
# CELL 7 | EDA — Text and Language Analysis
# Runs on train_df only (no test leakage in feature analysis).
# Analyses: 1, 3, 5, 6 + duplicate text + profile tier breakdown
# ============================================================

print('=' * 65)
print('  EDA — TEXT AND LANGUAGE ANALYSIS')
print('=' * 65)


# ── Analysis 1 | Language / script distribution ───────────
print('\n── Analysis 1: Language / Script Distribution ─────────────')

def detect_script(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 'empty'
    bengali = len(re.findall(r'[\u0980-\u09FF]', text))
    latin   = len(re.findall(r'[a-zA-Z]',        text))
    total   = bengali + latin
    if total == 0:
        return 'other'
    if bengali / total > 0.6:
        return 'bengali'
    elif latin  / total > 0.6:
        return 'latin'
    else:
        return 'mixed'

BANGLISH_MARKERS = [
    r'\b(ami|tumi|apni|valo|bhalo|sundor|khub|ektu|ki|ke|keno|ache|hobe|nai|nei)\b'
]

def refine_latin(text):
    text_lower = text.lower()
    return 'banglish' if any(
        re.search(p, text_lower) for p in BANGLISH_MARKERS
    ) else 'english'

train_df['script'] = train_df['review_text'].apply(detect_script)
latin_mask = train_df['script'] == 'latin'
train_df.loc[latin_mask, 'script'] = train_df.loc[
    latin_mask, 'review_text'
].apply(refine_latin)

script_counts = train_df['script'].value_counts()
script_pct    = train_df['script'].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({'Count': script_counts, 'Pct (%)': script_pct}).to_string())


# ── Analysis 3 | Transliteration inconsistency ────────────
print('\n── Analysis 3: Transliteration Inconsistency ───────────────')

SPELLING_CLUSTERS = {
    'good':    ['valo', 'bhalo', 'balo', 'vhalo'],
    'very':    ['khub', 'khob', 'kub', 'khoop'],
    'nice':    ['sundor', 'sundar', 'shundor', 'shundar'],
    'bad':     ['kharap', 'kharrap', 'khrap', 'kharaap'],
    'price':   ['dam', 'daam', 'darm', 'daam'],
    'fast':    ['druto', 'jaldi', 'tara tari', 'taratari'],
    'product': ['product', 'produk', 'protukt', 'pradakt'],
}

corpus = train_df['review_text'].dropna().str.lower()

print(f"  {'Concept':<10}  {'Variants & counts'}")
print('  ' + '─' * 55)
for concept, variants in SPELLING_CLUSTERS.items():
    counts = {
        v: int(corpus.str.contains(r'\b' + v + r'\b', regex=True).sum())
        for v in variants
    }
    found = {v: c for v, c in counts.items() if c > 0}
    if found:
        variant_str = '  '.join(f'{v}:{c:,}' for v, c in found.items())
        print(f"  {concept:<10}  {variant_str}")


# ── Analysis 5 | User language consistency ────────────────
print('\n── Analysis 5: User Language Consistency ───────────────────')

user_lang = (
    train_df.groupby(['user_id', 'script'])
    .size()
    .unstack(fill_value=0)
)
user_lang['dominant']    = user_lang.idxmax(axis=1)
user_lang['consistency'] = user_lang.iloc[:, :-1].max(axis=1) / \
                           user_lang.iloc[:, :-1].sum(axis=1)

print(f'  Consistency score distribution:')
print(user_lang['consistency'].describe().round(3).to_string())
mono = (user_lang['consistency'] > 0.9).sum()
print(f'\n  Monolingual users (>90% one script): '
      f'{mono:,} / {len(user_lang):,} '
      f'({mono/len(user_lang)*100:.1f}%)')
print(f'\n  Dominant script distribution:')
print(user_lang['dominant'].value_counts().to_string())


# ── Analysis 6 | Review length by script ──────────────────
print('\n── Analysis 6: Review Length by Script ─────────────────────')

train_df['text_len'] = train_df['review_text'].str.len()
len_by_script = train_df.groupby('script')['text_len'].describe().round(1)
print(len_by_script[['count', 'mean', '50%', '75%', 'max']].to_string())
print(f"\n  Empty reviews (len=0)   : {(train_df['text_len']==0).sum():,}")
print(f"  Very short (len < 5)    : {(train_df['text_len']<5).sum():,}")
print(f"  Substantial (len >= 20) : {(train_df['text_len']>=20).sum():,}")


# ── Duplicate text in non-empty reviews ───────────────────
print('\n── Duplicate Review Text (non-empty only) ──────────────────')

non_empty = train_df[train_df['review_text'].str.len() > 5]
dup_text      = non_empty.duplicated(subset=['review_text']).sum()
dup_user_text = non_empty.duplicated(subset=['user_id', 'review_text']).sum()
print(f'  Non-empty reviews             : {len(non_empty):,}')
print(f'  Duplicate texts               : {dup_text:,}  '
      f'({dup_text/len(non_empty)*100:.1f}%)')
print(f'  Same user + same text         : {dup_user_text:,}  '
      f'({dup_user_text/len(non_empty)*100:.1f}%)')


# ── Profile tier breakdown ────────────────────────────────
print('\n── Item Profile Tier Breakdown ─────────────────────────────')

item_text_coverage = (
    train_df.groupby('product_id')['review_text']
    .apply(lambda x: any(t.strip() != '' for t in x))
)
tier1 = item_text_coverage.sum()
tier3 = (~item_text_coverage).sum()
total_items = len(item_text_coverage)

print(f'  Total items                   : {total_items:,}')
print(f'  Tier 1 — has review text      : {tier1:,}  '
      f'({tier1/total_items*100:.1f}%)')
print(f'  Tier 3 — no text (zero vector): {tier3:,}  '
      f'({tier3/total_items*100:.1f}%)')
print('\n✅ EDA complete.')

In [ ]:
# ============================================================
# CELL 8 | Sparse matrix + index maps
# ============================================================

def build_sparse_matrix(train_df):
    users = train_df['user_id'].unique()
    items = train_df['product_id'].unique()

    user2idx = {u: i for i, u in enumerate(users)}
    item2idx = {it: i for i, it in enumerate(items)}
    idx2user = {i: u  for u, i in user2idx.items()}
    idx2item = {i: it for it, i in item2idx.items()}

    row  = train_df['user_id'].map(user2idx).values
    col  = train_df['product_id'].map(item2idx).values
    data = train_df['rating'].values.astype(np.float32)

    matrix = sp.csr_matrix(
        (data, (row, col)),
        shape=(len(users), len(items)),
        dtype=np.float32,
    )
    return matrix, user2idx, item2idx, idx2user, idx2item


matrix, user2idx, item2idx, idx2user, idx2item = build_sparse_matrix(train_df)
n_users = len(user2idx)
n_items = len(item2idx)

# Add integer index columns to train_df
train_df['user_idx'] = train_df['user_id'].map(user2idx)
train_df['item_idx'] = train_df['product_id'].map(item2idx)

density = matrix.nnz / (n_users * n_items)
print(f'✅ Matrix : {n_users:,} users × {n_items:,} items')
print(f'   Non-zeros : {matrix.nnz:,}')
print(f'   Density   : {density:.5%}')

In [ ]:
# ============================================================
# CELL 9 | Build item profiles — TF-IDF with 3-tier fallback
#
# Tier 1: mean TF-IDF of deduplicated review texts per item
# Tier 2: category string (items with no review text)
# Tier 3: zero vector (no text, no category)
#
# Method: mean of per-review vectors, not concat
# Vectorizer: char_wb 1-2gram — handles Banglish spelling variation
# ============================================================

MAX_FEATURES = 15_000

# ── Step 1: Build per-item document corpus ────────────────
# Deduplicate review texts per product first (removes copy-paste bias)
item_reviews = (
    train_df[train_df['review_text'].str.len() > 5]
    .drop_duplicates(subset=['product_id', 'review_text'])
    .groupby('product_id')['review_text']
    .apply(list)
    .to_dict()
)

item_categories = (
    train_df.groupby('product_id')['category']
    .agg(lambda x: x.mode().iat[0])
    .to_dict()
)

# ── Step 2: Fit TF-IDF on all available review texts ─────
all_texts = [
    text
    for reviews in item_reviews.values()
    for text in reviews
]

print(f'Fitting TF-IDF on {len(all_texts):,} review texts …')
t0 = time.time()

tfidf = TfidfVectorizer(
    max_features  = MAX_FEATURES,
    ngram_range   = (1, 2),
    analyzer      = 'char_wb',
    sublinear_tf  = True,
    strip_accents = None,     # preserve Bengali/Banglish unicode
    norm          = 'l2',
)
tfidf.fit(all_texts)
print(f'   Vocab size : {len(tfidf.vocabulary_):,}  |  done in {time.time()-t0:.1f}s')

# ── Step 3: Build item profile matrix ─────────────────────
# Shape: (n_items × max_features), aligned with item2idx
item_profile_matrix = np.zeros((n_items, MAX_FEATURES), dtype=np.float32)

tier_counts = {1: 0, 2: 0, 3: 0}

for pid, cf_idx in tqdm(item2idx.items(), desc='Building item profiles'):
    reviews = item_reviews.get(pid, [])

    if reviews:
        # Tier 1 — mean of individual review TF-IDF vectors
        vecs = tfidf.transform(reviews)          # (n_reviews × vocab)
        profile = np.asarray(vecs.mean(axis=0))  # (1 × vocab) → mean
        item_profile_matrix[cf_idx] = profile.flatten()
        tier_counts[1] += 1

    else:
        cat = str(item_categories.get(pid, ''))
        if cat and cat != 'Unknown':
            # Tier 2 — category string as fallback document
            vec = tfidf.transform([cat])
            item_profile_matrix[cf_idx] = np.asarray(vec.todense()).flatten()
            tier_counts[2] += 1
        else:
            # Tier 3 — zero vector (no signal)
            tier_counts[3] += 1

# L2-normalise all rows so cosine similarity = dot product
item_profile_matrix = normalize(item_profile_matrix, norm='l2')

print(f'\n✅ Item profile matrix: {item_profile_matrix.shape}')
print(f'   Tier 1 (review text) : {tier_counts[1]:,}')
print(f'   Tier 2 (category)    : {tier_counts[2]:,}')
print(f'   Tier 3 (zero vector) : {tier_counts[3]:,}')

In [ ]:
# ============================================================
# CELL 10 | Build user profiles — mean TF-IDF
#
# User profile = L2-normalised mean of item profile vectors
# for all items in the user's training history.
# ============================================================

print('Building user profiles …')
t0 = time.time()

user_profile_matrix = np.zeros((n_users, MAX_FEATURES), dtype=np.float32)

user_items_train = (
    train_df.groupby('user_id')['product_id']
    .apply(list)
    .to_dict()
)

n_cold = 0
for uid, u_idx in tqdm(user2idx.items(), desc='User profiles', leave=False):
    items_seen = user_items_train.get(uid, [])
    item_idxs  = [item2idx[p] for p in items_seen if p in item2idx]

    if not item_idxs:
        n_cold += 1
        continue

    # Mean of item profile vectors (already L2-normalised)
    profile = item_profile_matrix[item_idxs].mean(axis=0)
    user_profile_matrix[u_idx] = profile

# L2-normalise user profiles
user_profile_matrix = normalize(user_profile_matrix, norm='l2')

print(f'\n✅ User profile matrix  : {user_profile_matrix.shape}')
print(f'   Cold-start users     : {n_cold:,}  (zero profile)')
print(f'   Done in {time.time()-t0:.1f}s')

In [ ]:
# ============================================================
# CELL 11 | ContentBasedRecommender class
# ============================================================

class ContentBasedRecommender:
    """
    Content-based filtering using precomputed TF-IDF item and user profiles.
    score_all_items: one matrix multiply — no Python loop.
    """

    def fit(self, train_df, matrix=None, user2idx=None,
            item2idx=None, idx2item=None, **kwargs):
        self.user2idx  = user2idx
        self.item2idx  = item2idx
        self.idx2item  = idx2item
        self.user_seen = (
            train_df.groupby('user_id')['product_id']
            .apply(set).to_dict()
        )
        # Profiles built in Cells 9-10 — stored as class attributes
        self._item_profiles = item_profile_matrix   # (n_items × vocab)
        self._user_profiles = user_profile_matrix   # (n_users × vocab)
        print(f'  Item profiles : {self._item_profiles.shape}')
        print(f'  User profiles : {self._user_profiles.shape}')
        return self

    def score_all_items(self, user_id: str) -> np.ndarray:
        """
        Returns (n_items,) cosine similarity vector aligned with item2idx.
        Since both profiles are L2-normalised, cosine sim = dot product.
        """
        u_idx = self.user2idx.get(user_id, -1)
        if u_idx < 0:
            return None
        # (vocab,) @ (vocab × n_items) → (n_items,)
        return self._user_profiles[u_idx] @ self._item_profiles.T

    def recommend(self, user_id: str, k: int = 10) -> list:
        scores = self.score_all_items(user_id)
        if scores is None:
            return []
        seen = self.user_seen.get(user_id, set())
        for p in seen:
            ci = self.item2idx.get(p, -1)
            if ci >= 0:
                scores[ci] = -np.inf
        top_k = np.argsort(scores)[-k:][::-1]
        return [self.idx2item[i] for i in top_k if scores[i] > -np.inf]


print('✅ ContentBasedRecommender defined.')

In [ ]:
# ============================================================
# CELL 12 | Fit model
# ============================================================

shared_fit_args = dict(
    train_df = train_df,
    matrix   = matrix,
    user2idx = user2idx,
    item2idx = item2idx,
    idx2item = idx2item,
)

MODELS = {
    'Content-Based (TF-IDF, char 1-2gram)': ContentBasedRecommender(),
}

for name, model in MODELS.items():
    print(f'⏳  Fitting  {name}')
    t0 = time.time()
    model.fit(**shared_fit_args)
    print(f'    ✅ Done in {time.time()-t0:.1f}s')

print('\n✅ All models fitted.')

In [ ]:
# ============================================================
# CELL 13 | Evaluation — full test set, all K values
# ============================================================

K_VALUES = [5, 10, 15, 30, 50]

test_known  = test_df[test_df['user_id'].isin(user2idx)].copy()
test_users  = test_known['user_id'].values
test_truths = test_known['product_id'].values
N_TEST      = len(test_users)
print(f'Test users : {N_TEST:,}  |  K values = {K_VALUES}\n')


def vectorized_metrics_multi_k(effective_ranks_0idx, k_values):
    n          = len(effective_ranks_0idx)
    result     = {'N_eval': n}
    ranks_1idx = effective_ranks_0idx + 1
    for k in k_values:
        in_topk = effective_ranks_0idx < k
        result[f'Hit@{k}']  = float(np.mean(in_topk.astype(np.float32)))
        result[f'MRR@{k}']  = float(np.mean(np.where(in_topk, 1.0/ranks_1idx, 0.0)))
        result[f'NDCG@{k}'] = float(np.mean(np.where(in_topk, 1.0/np.log2(ranks_1idx+1), 0.0)))
    return result


def evaluate_model(model, test_users, test_truths,
                   k_values, item2idx, model_name='model'):
    n       = len(test_users)
    n_items = len(item2idx)
    effective_ranks = np.full(n, n_items, dtype=np.int32)
    truth_item_idx  = np.array(
        [item2idx.get(t, -1) for t in test_truths], dtype=np.int32
    )
    t0          = time.time()
    report_step = max(1, n // 20)

    for i, u in enumerate(test_users):
        t_idx = truth_item_idx[i]
        if t_idx < 0:
            continue
        scores = model.score_all_items(u)
        if scores is None:
            continue
        scores = scores.astype(np.float32, copy=True)
        for p in model.user_seen.get(u, set()):
            ci = item2idx.get(p, -1)
            if ci >= 0 and ci != t_idx:
                scores[ci] = -np.inf
        truth_score = scores[t_idx]
        if not np.isfinite(truth_score):
            continue
        effective_ranks[i] = int((scores > truth_score).sum())

        if (i+1) % report_step == 0 or (i+1) == n:
            elapsed = time.time() - t0
            rate    = (i+1) / max(elapsed, 1e-6)
            eta     = (n - i - 1) / rate
            print(f'    {(i+1)/n*100:5.1f}%  |  {i+1:,}/{n:,}  |  '
                  f'{rate:.0f} users/s  |  ETA {eta:.0f}s', end='\r')

    print(f'    100.0%  |  {n:,}/{n:,}  |  done in {time.time()-t0:.1f}s          ')
    return vectorized_metrics_multi_k(effective_ranks, k_values)


# ── Run ──────────────────────────────────────────────────
results = {}
for name, model in MODELS.items():
    print(f'\n⏳  Evaluating  {name}')
    results[name] = evaluate_model(
        model, test_users, test_truths, K_VALUES, item2idx, name
    )
    m       = results[name]
    summary = '  '.join(
        f'Hit@{k}={m[f"Hit@{k}"]:.4f}  NDCG@{k}={m[f"NDCG@{k}"]:.4f}'
        for k in K_VALUES
    )
    print(f'    {summary}')


# ── Tables ───────────────────────────────────────────────
print('\n' + '█'*65)
print(f'{"█  FULL EVALUATION RESULTS  █":^65}')
print('█'*65)

for k in K_VALUES:
    rows = sorted(
        [[name, m['N_eval'],
          round(m[f'Hit@{k}'],  4),
          round(m[f'MRR@{k}'],  4),
          round(m[f'NDCG@{k}'], 4)]
         for name, m in results.items()],
        key=lambda x: x[4], reverse=True
    )
    print(f"\n{'─'*65}\n  K = {k}\n{'─'*65}")
    print(tabulate(
        rows,
        headers=['Model', 'N Eval', f'Hit@{k}', f'MRR@{k}', f'NDCG@{k}'],
        tablefmt='fancy_grid', floatfmt='.4f'
    ))

print(f"\n{'─'*65}\n  NDCG SUMMARY\n{'─'*65}")
summary_rows = sorted(
    [[name] + [round(results[name][f'NDCG@{k}'], 4) for k in K_VALUES]
     for name in results],
    key=lambda r: sum(r[1:])/len(K_VALUES), reverse=True
)
print(tabulate(
    summary_rows,
    headers=['Model'] + [f'NDCG@{k}' for k in K_VALUES],
    tablefmt='fancy_grid', floatfmt='.4f'
))

In [ ]:
# ============================================================
# CELL 14 | Segmented evaluation by user script
#
# Evaluates NDCG@10 separately for Bengali-dominant,
# Banglish-dominant, and mixed/other users.
# Novel contribution: no prior RecSys paper has reported
# recommendation quality segmented by script type.
# ============================================================

print('── Segmented Evaluation by User Script ─────────────────────\n')

# user_lang built in Cell 7 Analysis 5
user_dominant_script = user_lang['dominant'].to_dict()

segments = ['bengali', 'banglish', 'english', 'mixed', 'empty']
seg_results = {}

for seg in segments:
    seg_users  = [
        u for u in test_users
        if user_dominant_script.get(u, 'unknown') == seg
    ]
    seg_truths = [
        test_truths[i]
        for i, u in enumerate(test_users)
        if user_dominant_script.get(u, 'unknown') == seg
    ]

    if len(seg_users) < 50:   # skip segments too small to be meaningful
        print(f'  {seg:<12} — skipped (only {len(seg_users)} users)')
        continue

    print(f'\n⏳  Segment: {seg}  ({len(seg_users):,} users)')
    seg_results[seg] = evaluate_model(
        model        = list(MODELS.values())[0],
        test_users   = np.array(seg_users),
        test_truths  = np.array(seg_truths),
        k_values     = [10],
        item2idx     = item2idx,
        model_name   = f'CB ({seg})',
    )

print(f"\n{'─'*55}")
print(f"  {'Segment':<14}  {'N Users':>8}  {'NDCG@10':>8}  {'Hit@10':>8}")
print(f"{'─'*55}")
for seg, m in seg_results.items():
    print(f"  {seg:<14}  {m['N_eval']:>8,}  {m['NDCG@10']:>8.4f}  {m['Hit@10']:>8.4f}")
print(f"{'─'*55}")
print('\n✅ Segmented evaluation complete.')

In [ ]:
# ============================================================
# CELL 15 | Sample recommendation demonstration
#
# Shows 3 users: one hit, one threshold user, one miss.
# Users selected by type with fixed seed — not cherry-picked.
# ============================================================

cb_model = list(MODELS.values())[0]
rng      = np.random.default_rng(42)
K_DEMO   = 10


def show_recommendation(model, user_id, train_df, test_df,
                         item2idx, idx2item, user_dominant_script, k=10):
    history = train_df[train_df['user_id'] == user_id][
        ['product_id', 'category', 'rating']
    ].sort_values('rating', ascending=False)

    truth_row  = test_df[test_df['user_id'] == user_id]
    truth_item = truth_row['product_id'].values[0] if len(truth_row) else None

    scores = model.score_all_items(user_id).copy()
    seen   = model.user_seen.get(user_id, set())
    for p in seen:
        ci = item2idx.get(p, -1)
        if ci >= 0:
            scores[ci] = -np.inf
    top_k_idx = np.argsort(scores)[-k:][::-1]
    recs = [idx2item[i] for i in top_k_idx]
    hit  = truth_item in recs if truth_item else False

    dominant_script = user_dominant_script.get(user_id, 'unknown')

    print('═' * 65)
    print(f'  User ID        : {user_id}')
    print(f'  Dominant script: {dominant_script}')
    print(f'  History ({len(history):,} interactions):')
    for _, row in history.head(5).iterrows():
        print(f'    [{row["rating"]:.0f}★]  {row["product_id"]}  '
              f'({row["category"]})')
    if len(history) > 5:
        print(f'    … and {len(history)-5} more')

    print(f'\n  Top-{k} Recommendations:')
    for rank, pid in enumerate(recs, 1):
        cat    = train_df[train_df['product_id'] == pid]['category'].iloc[0] \
                 if pid in train_df['product_id'].values else 'unknown'
        marker = '  ← GROUND TRUTH ✅' if pid == truth_item else ''
        print(f'    {rank:>2}. {pid}  ({cat}){marker}')

    if truth_item and not hit:
        truth_cat = train_df[train_df['product_id'] == truth_item]['category'] \
                    .iloc[0] if truth_item in train_df['product_id'].values else 'unknown'
        print(f'\n  Ground truth   : {truth_item}  ({truth_cat})  ❌ not in top-{k}')
    print()


# ── Find representative users ────────────────────────────
interaction_counts = train_df['user_id'].value_counts()

# 1. Hit user — ground truth appears in top-10
hit_user = None
for uid in rng.choice(test_known['user_id'].values,
                       min(500, len(test_known)), replace=False):
    scores = cb_model.score_all_items(uid)
    if scores is None:
        continue
    truth = test_known[test_known['user_id'] == uid]['product_id'].values[0]
    t_idx = item2idx.get(truth, -1)
    if t_idx >= 0 and int((scores > scores[t_idx]).sum()) < K_DEMO:
        hit_user = uid
        break

# 2. Threshold user — 10-15 interactions
threshold_candidates = [
    u for u in interaction_counts[
        interaction_counts.between(10, 15)
    ].index if u in user2idx and u in test_known['user_id'].values
]
threshold_user = rng.choice(threshold_candidates, 1)[0] \
                 if threshold_candidates else None

# 3. Miss user — ground truth not in top-10
miss_user = None
for uid in rng.choice(test_known['user_id'].values,
                       min(500, len(test_known)), replace=False):
    scores = cb_model.score_all_items(uid)
    if scores is None:
        continue
    truth = test_known[test_known['user_id'] == uid]['product_id'].values[0]
    t_idx = item2idx.get(truth, -1)
    if t_idx >= 0 and int((scores > scores[t_idx]).sum()) >= K_DEMO:
        miss_user = uid
        break

# ── Display ───────────────────────────────────────────────
user_dominant_script_map = user_lang['dominant'].to_dict()

demo_users = [
    ('HIT USER',       hit_user),
    ('THRESHOLD USER', threshold_user),
    ('MISS USER',      miss_user),
]

for label, uid in demo_users:
    if uid is None:
        print(f'[{label}] — no suitable user found')
        continue
    print(f'\n[{label}]')
    show_recommendation(
        cb_model, uid, train_df, test_df,
        item2idx, idx2item, user_dominant_script_map, k=K_DEMO
    )

In [ ]:
# ============================================================
# CELL 16 | Save all results to CSV
# ============================================================

# ── Full evaluation results ───────────────────────────────
rows = []
for name, m in results.items():
    row = {'Model': name, 'N_eval': m['N_eval']}
    for k in K_VALUES:
        row[f'Hit@{k}']  = m[f'Hit@{k}']
        row[f'MRR@{k}']  = m[f'MRR@{k}']
        row[f'NDCG@{k}'] = m[f'NDCG@{k}']
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv('cb_benchmark_results.csv', index=False)
print('✅ Full results saved to cb_benchmark_results.csv')
print(results_df.to_string(index=False))

# ── Segmented results ─────────────────────────────────────
if seg_results:
    seg_rows = []
    for seg, m in seg_results.items():
        seg_rows.append({
            'Segment':  seg,
            'N_eval':   m['N_eval'],
            'NDCG@10':  m['NDCG@10'],
            'Hit@10':   m['Hit@10'],
            'MRR@10':   m['MRR@10'],
        })
    seg_df = pd.DataFrame(seg_rows)
    seg_df.to_csv('cb_segmented_results.csv', index=False)
    print('\n✅ Segmented results saved to cb_segmented_results.csv')
    print(seg_df.to_string(index=False))